# 00 · Setup verification

Run this first in any new Colab session. It confirms the environment, the checkpoint layer, and the resume behaviour before you spend hours on a real run.


In [ ]:
# Bootstrap -- see environment/colab_bootstrap.md for the full version
import os
os.environ.setdefault('AFS_DATA_ROOT', '/content/drive/MyDrive/afs-data')
os.environ.setdefault('AFS_SCRATCH', '/content/afs-scratch')
from afs.paths import Paths
from afs.config import resolve_experiment
from afs.pipeline import run_experiment
paths = Paths.create()
print('data root:', paths.data_root)


In [ ]:
!pytest -q

## Verify crash-resume

Simulates a Colab disconnect mid-loop and confirms that only the incomplete items are recomputed. If this fails, do not start a long run.


In [ ]:
from afs.checkpoint import ResumableLoop
import shutil; shutil.rmtree('/tmp/rt', ignore_errors=True)

def flaky(i):
    if i == 55: raise RuntimeError('simulated disconnect')
    return {'value': i}

loop = ResumableLoop('/tmp/rt', shard_size=10)
try: loop.run(range(100), flaky, key=int, progress=False)
except RuntimeError as e: print('crashed:', e)
print('persisted:', len(loop.completed_ids()))

df = ResumableLoop('/tmp/rt', shard_size=10).run(range(100), lambda i: {'value': i}, key=int, progress=False)
print('after resume:', len(df), 'unique:', df['item_id'].nunique())

In [ ]:
# End-to-end smoke run on synthetic data
res = run_experiment(resolve_experiment('00_smoke'), paths)
for arm, v in res['arms'].items():
    print(f"{arm:20s} auc={v['roc_auc']:.4f}  evasion={v.get('evasion',{}).get('median_cost')}")